# Validação 13 — Qualidade e integridade do artigo

## Goal

Comprovar que o perfil de qualidade registra verificações externas de identidade, retratação, protocolo aplicável, desenho declarado, texto completo e dados relacionados, sem converter ausência de registro em certeza.

## Setup

Crossref fornece identidade e atualizações editoriais, incluindo registros incorporados do Retraction Watch. DataCite é consultado somente por datasets com relação explícita e qualificadora ao DOI. ClinicalTrials.gov só é usado quando o artigo é um ensaio clínico e contém um identificador NCT. `NOT_FOUND` significa apenas que a consulta não localizou o item; não demonstra inexistência.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import os
import sys

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from fatofake import (
    ClinicalTrialsClient,
    CrossrefClient,
    DataCiteClient,
    PmcClient,
    PubMedClient,
    QualityLevel,
    StudyDesign,
    ValidationStatus,
    prepare_search_plan,
    retrieve_article_content,
    search_pubmed,
    validate_analysis_input,
    validate_article_quality,
)

executed_at = datetime.now(timezone.utc).isoformat()
print(f'Execução UTC: {executed_at}')

Execução UTC: 2026-09-25T12:20:25.400622+00:00


## Steps

### 1. Recuperar o artigo e seu conteúdo

Usamos o mesmo artigo real das etapas anteriores para que o novo perfil possa substituir diretamente o perfil manual do notebook 12.

In [2]:
class PmidQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ['33431520[pmid]']

analysis_input = validate_analysis_input(
    'Beber café pode alterar o risco de câncer de próstata.'
)
search_plan = prepare_search_plan(analysis_input, PmidQueryPlanner())
pubmed_result = search_pubmed(
    search_plan,
    PubMedClient(email=os.getenv('NCBI_EMAIL'), api_key=os.getenv('NCBI_API_KEY')),
    max_results_per_query=1,
)
publication = pubmed_result.publications[0]
content = retrieve_article_content(
    publication,
    PmcClient(email=os.getenv('NCBI_EMAIL'), api_key=os.getenv('NCBI_API_KEY')),
)
print(f'PMID: {publication.pmid} | DOI: {publication.doi}')
print(publication.title)

PMID: 33431520 | DOI: 10.1136/bmjopen-2020-038902
Coffee consumption and risk of prostate cancer: a systematic review and meta-analysis.


### 2. Consultar as fontes externas

Cada resultado mantém status, explicação e URL da fonte. Uma falha de serviço vira `UNKNOWN` e não interrompe nem inventa o restante do perfil.

In [3]:
quality_report = validate_article_quality(
    publication,
    content,
    CrossrefClient(email=os.getenv('CROSSREF_EMAIL')),
    DataCiteClient(),
    ClinicalTrialsClient(),
)
check_rows = [
    {
        'check': item.name,
        'status': item.status.value,
        'summary': item.summary,
        'source_url': item.source_url,
    }
    for item in quality_report.checks
]
pprint(check_rows)

[{'check': 'study_design',
  'source_url': 'https://pubmed.ncbi.nlm.nih.gov/33431520/',
  'status': 'CONFIRMED',
  'summary': 'Desenho identificado de forma conservadora: '
             'SYSTEMATIC_REVIEW_META_ANALYSIS.'},
 {'check': 'identity',
  'source_url': 'https://doi.org/10.1136/bmjopen-2020-038902',
  'status': 'CONFIRMED',
  'summary': 'DOI resolvido e título compatível entre PubMed e Crossref.'},
 {'check': 'retraction',
  'source_url': 'https://doi.org/10.1136/bmjopen-2020-038902',
  'status': 'NOT_FOUND',
  'summary': 'Nenhuma retratação foi localizada no registro consultado; '
             'ausência não prova inexistência.'},
 {'check': 'data_availability',
  'source_url': 'https://api.datacite.org/',
  'status': 'NOT_FOUND',
  'summary': 'Nenhum dataset com relação qualificadora foi localizado; isso '
             'não prova que os dados não existam.'},
 {'check': 'trial_registration',
  'source_url': 'https://clinicaltrials.gov/',
  'status': 'NOT_APPLICABLE',
  'summary

### 3. Produzir o perfil consumido pela síntese

Metadados e integridade ajudam a detectar problemas objetivos, mas não substituem avaliação de risco de viés. Sem essa avaliação, a qualidade não é promovida acima de `UNCLEAR`.

In [4]:
quality_profile = quality_report.to_quality_profile(publication)
pprint({
    'pmid': quality_profile.pmid,
    'study_design': quality_profile.study_design,
    'quality_level': quality_profile.level.value,
    'quality_weight': quality_profile.weight,
    'related_datasets': [dataset.doi for dataset in quality_report.datasets],
    'trial_registrations': [trial.nct_id for trial in quality_report.trial_registrations],
    'rationale': quality_profile.rationale,
})

{'pmid': '33431520',
 'quality_level': 'UNCLEAR',
 'quality_weight': 0.25,
 'rationale': 'Metadados externos foram verificados, mas a qualidade permanece '
              'não esclarecida até uma avaliação estruturada de risco de viés.',
 'related_datasets': [],
 'study_design': 'SYSTEMATIC_REVIEW_META_ANALYSIS',
 'trial_registrations': []}


## Checks

As verificações confirmam proveniência completa, desenho conservador, aplicação condicional do ClinicalTrials.gov e manutenção explícita das lacunas.

In [5]:
statuses = {item.name: item.status for item in quality_report.checks}
assert quality_report.study_design is StudyDesign.SYSTEMATIC_REVIEW_META_ANALYSIS
assert statuses['study_design'] is ValidationStatus.CONFIRMED
assert statuses['identity'] is ValidationStatus.CONFIRMED
assert statuses['retraction'] in {ValidationStatus.CONFIRMED, ValidationStatus.NOT_FOUND}
assert statuses['trial_registration'] is ValidationStatus.NOT_APPLICABLE
assert statuses['data_availability'] in {ValidationStatus.CONFIRMED, ValidationStatus.NOT_FOUND}
assert statuses['full_text'] is ValidationStatus.CONFIRMED
assert all(item.source_url for item in quality_report.checks)
assert quality_profile.level in {QualityLevel.UNCLEAR, QualityLevel.LOW}
assert quality_profile.pmid == publication.pmid

print(
    f'Validação aprovada: {len(quality_report.checks)} verificações rastreáveis; '
    f'qualidade final {quality_profile.level.value}.'
)

Validação aprovada: 6 verificações rastreáveis; qualidade final UNCLEAR.


## Next Steps

A etapa estará validada quando todas as células forem executadas sem erros. A próxima etapa será aplicar um instrumento de risco de viés adequado ao desenho detectado — por exemplo, AMSTAR 2 para revisões sistemáticas — antes de graduar a qualidade científica.